# AI Agentic YouTube Comment Sentiment Analysis & SQL Reporting System


### What this notebook does
1. Takes a YouTube channel as input (handle / URL / channel ID)
2. Uses a **Hugging Face smolagents** agent to orchestrate the full workflow
3. Fetches the latest 10 videos and extracts comments via YouTube Data API
4. Stores everything in a MySQL (TiDB) database
5. Runs Hugging Face sentiment analysis on every comment
6. Generates video-wise and channel-wise sentiment reports
7. Exports CSV files + SQL dump


In [ ]:
# Installing all required libraries
!pip install smolagents google-api-python-client mysql-connector-python transformers torch pandas -q
print("All libraries installed!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.7 MB/s eta 0:00:00
All libraries installed!


## API Keys & Database Config

In [1]:
from getpass import getpass

YOUTUBE_API_KEY = getpass("Enter YouTube API Key: ")
HF_TOKEN = getpass("Enter Hugging Face Token: ")

DB_CONFIG = {
    "host"    : getpass("Enter DB Host: "),
    "port"    : 4000,
    "user"    : getpass("Enter DB User: "),
    "password": getpass("Enter DB Password: "),
    "database": "project"
}

print("Credentials loaded securely!")

Enter YouTube API Key: ··········
Enter Hugging Face Token: ··········
Enter DB Host: ··········
Enter DB User: ··········
Enter DB Password: ··········
Credentials loaded securely!


In [ ]:




import re
import pandas as pd
import mysql.connector
from googleapiclient.discovery import build
from transformers import pipeline
from smolagents import tool, CodeAgent, InferenceClientModel

# ── YouTube API  ──────────────────────────────────────
youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
print("YouTube api ")

# ── Hugging Face Sentiment Model ────────────────────────────

print("Loading sentiment model (this may take a minute)...")
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    token=HF_TOKEN
)
print("Sentiment model ready!")


YouTube client ready!
Loading sentiment model (this may take a minute)...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


Sentiment model ready!


##  Createing Database Tables

In [ ]:

# Database Tables
# Three tables: channels > videos > comments


def create_tables():
    conn = mysql.connector.connect(**DB_CONFIG)
    cur  = conn.cursor()

    # Drop in reverse FK order so we start fresh every run
    cur.execute("DROP TABLE IF EXISTS comments")
    cur.execute("DROP TABLE IF EXISTS videos")
    cur.execute("DROP TABLE IF EXISTS channels")

    # Table 1: channels
    cur.execute("""
        CREATE TABLE channels (
            channel_id       VARCHAR(50)  PRIMARY KEY,
            channel_title    VARCHAR(255),
            description      TEXT,
            subscriber_count BIGINT,
            video_count      INT,
            view_count       BIGINT,
            created_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    # Table 2: videos
    cur.execute("""
        CREATE TABLE videos (
            video_id      VARCHAR(50)  PRIMARY KEY,
            channel_id    VARCHAR(50),
            title         VARCHAR(500),
            description   TEXT,
            published_at  DATETIME,
            view_count    BIGINT,
            like_count    BIGINT,
            comment_count BIGINT,
            FOREIGN KEY (channel_id) REFERENCES channels(channel_id)
        )
    """)

    # Table 3: comments  (videos)
    cur.execute("""
        CREATE TABLE comments (
            comment_id      VARCHAR(100) PRIMARY KEY,
            video_id        VARCHAR(50),
            author_name     VARCHAR(255),
            comment_text    TEXT,
            like_count      INT,
            published_at    DATETIME,
            sentiment_label VARCHAR(20),
            sentiment_score FLOAT,
            FOREIGN KEY (video_id) REFERENCES videos(video_id)
        )
    """)

    conn.commit()
    cur.close()
    conn.close()
    print("All three tables created successfully!")

create_tables()


All three tables created successfully!


##Define smolagents Tools

In [ ]:

#  Createing the 4 Steps (Tools) for the Agent
#
#
# The AI agent will follow these 4 steps one by one.
#
#   STEP 1 - Find the YouTube channel and save its info
#   STEP 2 - Get the latest 10 videos from that channel
#   STEP 3 - Get comments from each video + check positive/negative
#   STEP 4 - Make a report and save it as CSV files
#

#


from smolagents import tool



# STEP 1 - Find the channel and save it to the database

@tool
def resolve_and_save_channel(channel_input: str) -> str:
    """Find the YouTube channel using the handle, URL or ID.
    Save the channel details to the database.
    Return the channel_id and uploads_playlist_id for the next step.

    Args:
        channel_input: Channel handle like @MrBeast, full URL, or channel ID.

    Returns:
        SUCCESS|channel_id|channel_title|uploads_playlist_id  or  ERROR|reason
    """

    # We start with no channel_id. We will find it below.
    channel_id = None

    # ---- If the user gave a full YouTube link ----------------
    if "youtube.com" in channel_input:

        if "/@" in channel_input:


            handle = "@" + channel_input.split("/@")[1].split("/")[0]
            channel_input = handle

        elif "/channel/" in channel_input:


            channel_id = channel_input.split("/channel/")[1].split("/")[0]

    # ---- If we still don't have a channel_id, search for it --
    if channel_id is None:

        if channel_input.startswith("UC"):
            # If it already starts with "UC" it IS the channel ID, use it directly
            channel_id = channel_input

        else:
            #  YouTube for the channel (like typing it in the search bar)
            search_result = youtube.search().list(
                part="snippet",
                q=channel_input,    # what we are searching for
                type="channel",     # we only want channels, not videos
                maxResults=1        # just the top 1 result
            ).execute()

            # If YouTube found something, get the channel ID from the result
            if search_result.get("items"):
                channel_id = search_result["items"][0]["snippet"]["channelId"]

    # ---- If we still have no channel_id, the channel was not found
    if not channel_id:
        return "ERROR|Channel not found. Please check the name or link and try again."

    # ---- Asking YouTube for full information about this channel ----
    channel_info = youtube.channels().list(
        part="snippet,statistics,contentDetails",
        id=channel_id
    ).execute()

    # If YouTube gives us nothing back, something went wrong
    if not channel_info.get("items"):
        return "ERROR|Could not get channel details from YouTube API."

    # ---- Pull out the data we need ----
    ch = channel_info["items"][0]

    channel_name        = ch["snippet"]["title"]
    description         = ch["snippet"].get("description", "")[:1000]  # first 1000 characters
    subscribers         = int(ch["statistics"].get("subscriberCount", 0))
    total_videos        = int(ch["statistics"].get("videoCount", 0))
    total_views         = int(ch["statistics"].get("viewCount", 0))
    uploads_playlist_id = ch["contentDetails"]["relatedPlaylists"]["uploads"]
    # uploads_playlist_id is like a folder that holds all uploaded videos

    # ---- Saveing channel info to the database ----
    connection = mysql.connector.connect(**DB_CONFIG)
    cursor     = connection.cursor()

    # INSERT adds a new row

    cursor.execute("""
        INSERT INTO channels
            (channel_id, channel_title, description, subscriber_count, video_count, view_count)
        VALUES
            (%s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            channel_title    = VALUES(channel_title),
            subscriber_count = VALUES(subscriber_count),
            view_count       = VALUES(view_count)
    """, (channel_id, channel_name, description, subscribers, total_videos, total_views))

    connection.commit()    # save the changes to the database
    cursor.close()         # we are done with the cursor
    connection.close()     # we are done with the connection

    print("Channel saved to database!")
    print("  Name        :", channel_name)
    print("  Subscribers :", f"{subscribers:,}")


    return f"SUCCESS|{channel_id}|{channel_name}|{uploads_playlist_id}"






All 4 steps (tools) are ready!

  STEP 1 -> resolve_and_save_channel    finds the channel
  STEP 2 -> fetch_and_save_videos       gets 10 latest videos
  STEP 3 -> fetch_analyze_save_comments gets comments + checks positive/negative
  STEP 4 -> generate_sentiment_report   makes the final report + CSV files


In [ ]:


# STEP 2 - Get the latest 10 videos and save them

@tool
def fetch_and_save_videos(channel_id: str, uploads_playlist_id: str) -> str:
    """Get the 10 most recent videos from the channel.
    Save each video title, views, likes, etc. to the database.

    Args:
        channel_id: The channel ID from Step 1.
        uploads_playlist_id: The uploads playlist ID from Step 1.

    Returns:
        SUCCESS|video_id1,video_id2,...|N videos saved  or  ERROR|reason
    """

    # Asking YouTube for the latest 10 videos from the uploads playlist
    playlist_result = youtube.playlistItems().list(
        part="snippet,contentDetails",
        playlistId=uploads_playlist_id,
        maxResults=10    # we only want the 10 most recent videos
    ).execute()

    # This list will collect all the video IDs
    video_ids = []

    # Open one database connection for all 10 videos
    connection = mysql.connector.connect(**DB_CONFIG)
    cursor     = connection.cursor()

    # Go through each video in the result one by one
    for item in playlist_result.get("items", []):

        # Get the video ID from this playlist item
        video_id = item["contentDetails"]["videoId"]

        # Asking YouTube for the full details of this video (views, likes, etc.)
        video_details = youtube.videos().list(
            part="statistics,snippet",
            id=video_id
        ).execute()

        # If YouTube gave us nothing for this video, skip it
        if not video_details.get("items"):
            continue

        video = video_details["items"][0]

        # Convert the date from YouTube format "2024-01-15T10:30:00Z"
        # to a clean database format "2024-01-15 10:30:00"
        published_date = video["snippet"]["publishedAt"].replace("Z", "").replace("T", " ")[:19]

        # Save this video to the database
        cursor.execute("""
            INSERT INTO videos
                (video_id, channel_id, title, description,
                 published_at, view_count, like_count, comment_count)
            VALUES
                (%s, %s, %s, %s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                view_count = VALUES(view_count),
                like_count = VALUES(like_count)
        """, (
            video_id,
            channel_id,
            video["snippet"]["title"],
            video["snippet"].get("description", "")[:1000],
            published_date,
            int(video["statistics"].get("viewCount",    0)),
            int(video["statistics"].get("likeCount",    0)),
            int(video["statistics"].get("commentCount", 0))
        ))

        # Add this video ID to our list
        video_ids.append(video_id)
        print("  Video saved:", video["snippet"]["title"][:60])

    # Save all changes and close the database
    connection.commit()
    cursor.close()
    connection.close()

    print(f"\n{len(video_ids)} videos saved to database!")

    # Join all video IDs with commas for the next step

    all_ids = ",".join(video_ids)
    return f"SUCCESS|{all_ids}|{len(video_ids)} videos saved"






In [ ]:
# STEP 3 - Get comments, check positive/negative, save to DB
@tool
def fetch_analyze_save_comments(video_ids_str: str) -> str:
    """For every video, get the comments, check if each comment is
    POSITIVE or NEGATIVE using the Hugging Face model, and save
    everything to the comments table in the database.

    Args:
        video_ids_str: Comma-separated video IDs from Step 2.
                       Example: "abc123,def456,ghi789"

    Returns:
        SUCCESS|total|positive|negative|neutral  or  ERROR|reason
    """

    # Split "abc123,def456" into a Python list: ["abc123", "def456"]
    video_list = [v.strip() for v in video_ids_str.split(",") if v.strip()]

    # Open one database connection for all the comments
    connection = mysql.connector.connect(**DB_CONFIG)
    cursor     = connection.cursor()

    # These counters keep track of totals across all videos
    total_comments    = 0
    positive_comments = 0
    negative_comments = 0
    neutral_comments  = 0

    # Go through each video one by one
    for video_id in video_list:

        print(f"\nGetting comments for video: {video_id}")

        # Collecting all comments for this video in this list
        comments_list = []

        try:
            # Ask YouTube for the comments on this video
            comment_result = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=100,           # YouTube gives 100 per page max
                order="relevance",        # most relevant comments first
                textFormat="plainText"    # plain text, no HTML tags
            ).execute()

            # Save each comment from the first page
            for item in comment_result.get("items", []):
                c = item["snippet"]["topLevelComment"]["snippet"]
                comments_list.append({
                    "comment_id"  : item["id"],
                    "author_name" : c.get("authorDisplayName", "Unknown"),
                    "comment_text": c.get("textDisplay", ""),
                    "like_count"  : int(c.get("likeCount", 0)),
                    "published_at": c["publishedAt"].replace("Z", "").replace("T", " ")[:19]
                })

            # If there are more pages of comments, keep getting them
            # (We stop at 200 comments per video so it doesn't take too long)
            while "nextPageToken" in comment_result and len(comments_list) < 200:
                comment_result = youtube.commentThreads().list(
                    part="snippet",
                    videoId=video_id,
                    maxResults=100,
                    order="relevance",
                    textFormat="plainText",
                    pageToken=comment_result["nextPageToken"]    # go to the next page
                ).execute()
                for item in comment_result.get("items", []):
                    c = item["snippet"]["topLevelComment"]["snippet"]
                    comments_list.append({
                        "comment_id"  : item["id"],
                        "author_name" : c.get("authorDisplayName", "Unknown"),
                        "comment_text": c.get("textDisplay", ""),
                        "like_count"  : int(c.get("likeCount", 0)),
                        "published_at": c["publishedAt"].replace("Z","").replace("T"," ")[:19]
                    })

        except Exception as error:
            # Some videos have comments turned off — that is okay, just skip them
            print("  Comments are turned off for this video:", str(error)[:80])
            continue    # move on to the next video

        print(f"  Found {len(comments_list)} comments — checking positive/negative now...")

        # ---- Checking each comment: positive or negative? ----
        for comment in comments_list:

            text = comment["comment_text"]


            clean_text = re.sub(r"[\x00-\x1f\x7f]", "", text).strip()
            clean_text = clean_text[:512]    # use only the first 512 characters

            if not clean_text:
                # If the comment has nothing in it, call it NEUTRAL
                label = "NEUTRAL"
                score = 0.5
            else:
                try:
                    # This is where the Hugging Face AI checks the comment!
                    #
                    result = sentiment_model(clean_text)[0]
                    label  = result["label"]              # "POSITIVE" or "NEGATIVE"
                    score  = round(result["score"], 4)    # how confident the model is
                except Exception:
                    # If the model has an error on this comment, call it NEUTRAL
                    label = "NEUTRAL"
                    score = 0.5

            # Save this comment and its sentiment label to the database
            cursor.execute("""
                INSERT INTO comments
                    (comment_id, video_id, author_name, comment_text,
                     like_count, published_at, sentiment_label, sentiment_score)
                VALUES
                    (%s, %s, %s, %s, %s, %s, %s, %s)
                ON DUPLICATE KEY UPDATE
                    sentiment_label = VALUES(sentiment_label),
                    sentiment_score = VALUES(sentiment_score)
            """, (
                comment["comment_id"],
                video_id,
                comment["author_name"],
                comment["comment_text"][:5000],   # keep max 5000 characters in the DB
                comment["like_count"],
                comment["published_at"],
                label,
                score
            ))


            total_comments += 1
            if   label == "POSITIVE": positive_comments += 1
            elif label == "NEGATIVE": negative_comments += 1
            else:                     neutral_comments  += 1

        # Save after finishing each video (good habit!)
        connection.commit()
        print(f"  Done! Running total so far: {total_comments} comments")

    # All videos are done — close the database connection
    cursor.close()
    connection.close()

    print("\nAll comments processed!")
    print("  Total    :", total_comments)
    print("  Positive :", positive_comments)
    print("  Negative :", negative_comments)
    print("  Neutral  :", neutral_comments)

    return f"SUCCESS|{total_comments} total|{positive_comments} positive|{negative_comments} negative|{neutral_comments} neutral"

In [ ]:

# STEP 4 - Make the final report and save as CSV files

@tool
def generate_sentiment_report(channel_id: str) -> str:
    """Read all data from the database and create three CSV reports:
    1. video_sentiment_report.csv  - one row per video
    2. channel_summary_report.csv  - overall channel totals
    3. all_comments_with_sentiment.csv - every single comment

    Args:
        channel_id: The channel ID from Step 1.

    Returns:
        A nicely formatted text summary of the sentiment results.
    """

    # Open the database to read all the data we saved in Steps 1-3
    connection = mysql.connector.connect(**DB_CONFIG)

    # ---- Report 1: Video-by-video breakdown ----
    # This SQL query counts how many positive/negative/neutral comments
    # each video got. One row = one video.
    video_report = pd.read_sql("""
        SELECT
            v.video_id,
            v.title,
            v.view_count,
            v.like_count,

            COUNT(c.comment_id) AS total_comments,

            SUM(CASE WHEN c.sentiment_label = 'POSITIVE' THEN 1 ELSE 0 END) AS positive_count,
            SUM(CASE WHEN c.sentiment_label = 'NEGATIVE' THEN 1 ELSE 0 END) AS negative_count,
            SUM(CASE WHEN c.sentiment_label = 'NEUTRAL'  THEN 1 ELSE 0 END) AS neutral_count,

            ROUND(AVG(c.sentiment_score), 4) AS avg_sentiment_score,

            ROUND(
                SUM(CASE WHEN c.sentiment_label = 'POSITIVE' THEN 1 ELSE 0 END) * 100.0
                / NULLIF(COUNT(c.comment_id), 0),
            2) AS positive_pct

        FROM videos v
        LEFT JOIN comments c ON v.video_id = c.video_id
        WHERE v.channel_id = %s
        GROUP BY v.video_id, v.title, v.view_count, v.like_count
        ORDER BY v.published_at DESC
    """, connection, params=(channel_id,))

    video_report.to_csv("video_sentiment_report.csv", index=False)
    print("File saved: video_sentiment_report.csv")

    # ---- Report 2: Whole channel summary ----
    channel_summary = pd.read_sql("""
        SELECT
            ch.channel_title,
            ch.subscriber_count,
            COUNT(DISTINCT v.video_id)   AS videos_analyzed,
            COUNT(c.comment_id)           AS total_comments,
            SUM(CASE WHEN c.sentiment_label = 'POSITIVE' THEN 1 ELSE 0 END) AS positive_comments,
            SUM(CASE WHEN c.sentiment_label = 'NEGATIVE' THEN 1 ELSE 0 END) AS negative_comments,
            SUM(CASE WHEN c.sentiment_label = 'NEUTRAL'  THEN 1 ELSE 0 END) AS neutral_comments,
            ROUND(AVG(c.sentiment_score), 4) AS avg_sentiment_score,
            ROUND(
                SUM(CASE WHEN c.sentiment_label = 'POSITIVE' THEN 1 ELSE 0 END) * 100.0
                / NULLIF(COUNT(c.comment_id), 0),
            2) AS positive_pct

        FROM channels ch
        LEFT JOIN videos   v ON ch.channel_id = v.channel_id
        LEFT JOIN comments c ON v.video_id    = c.video_id
        WHERE ch.channel_id = %s
        GROUP BY ch.channel_id, ch.channel_title, ch.subscriber_count
    """, connection, params=(channel_id,))

    channel_summary.to_csv("channel_summary_report.csv", index=False)
    print("File saved: channel_summary_report.csv")

    # ---- Report 3: Every comment with its sentiment label ----
    all_comments = pd.read_sql("""
        SELECT
            c.comment_id,
            v.title         AS video_title,
            c.author_name,
            c.comment_text,
            c.like_count,
            c.sentiment_label,
            c.sentiment_score,
            c.published_at
        FROM comments c
        JOIN videos v ON c.video_id = v.video_id
        WHERE v.channel_id = %s
        ORDER BY c.published_at DESC
    """, connection, params=(channel_id,))

    all_comments.to_csv("all_comments_with_sentiment.csv", index=False)
    print(f"File saved: all_comments_with_sentiment.csv  ({len(all_comments)} rows)")

    connection.close()

    # ---- If there is no data at all, something went wrong ----
    if channel_summary.empty:
        return "ERROR|No data found. Make sure Steps 1, 2, and 3 ran successfully first."

    # ---- Pull the numbers we need for the printed summary ----
    row = channel_summary.iloc[0]   # get the one and only row

    total   = int(row["total_comments"]    or 0)
    pos_c   = int(row["positive_comments"] or 0)
    neg_c   = int(row["negative_comments"] or 0)
    neu_c   = int(row["neutral_comments"]  or 0)

    # Calculate percentages (avoid dividing by zero if total is 0)
    pos_pct = round(pos_c * 100 / total, 2) if total > 0 else 0
    neg_pct = round(neg_c * 100 / total, 2) if total > 0 else 0
    neu_pct = round(neu_c * 100 / total, 2) if total > 0 else 0

    # Decide the overall verdict based on the positive percentage
    if   pos_pct >= 70:
        verdict = "VERY POSITIVE  - The audience absolutely loves this channel!"
    elif pos_pct >= 50:
        verdict = "MOSTLY POSITIVE  - Viewers generally enjoy this channel."
    elif pos_pct >= 30:
        verdict = "MIXED  - Some viewers like it, some do not."
    else:
        verdict = "MOSTLY NEGATIVE  - Viewers are unhappy with this channel."

    # Build and print the final summary
    final_report = f"""
============================================================
         YOUTUBE CHANNEL SENTIMENT REPORT
============================================================

  Channel Name    : {row['channel_title']}
  Subscribers     : {int(row['subscriber_count']):,}
  Videos Checked  : {int(row['videos_analyzed'])}
  Total Comments  : {total}

  ---- Sentiment Breakdown ----
  Positive  : {pos_c:>5} comments  ({pos_pct}%)
  Negative  : {neg_c:>5} comments  ({neg_pct}%)
  Neutral   : {neu_c:>5} comments  ({neu_pct}%)

  Avg Confidence  : {row['avg_sentiment_score']}
  Verdict         : {verdict}

  ---- Files Saved ----
  video_sentiment_report.csv
  channel_summary_report.csv
  all_comments_with_sentiment.csv

============================================================
"""

    print(final_report)
    return final_report


# ---- Quick confirmation that all 4 tools are ready ----
print("All 4 steps (tools) are ready!")
print()
print("  STEP 1 -> resolve_and_save_channel    finds the channel")
print("  STEP 2 -> fetch_and_save_videos       gets 10 latest videos")
print("  STEP 3 -> fetch_analyze_save_comments gets comments + checks positive/negative")
print("  STEP 4 -> generate_sentiment_report   makes the final report + CSV files")


##  Createing the smolagents Agent

In [ ]:
# ============================================================
#  Createing the Hugging Face smolagents Agent
#
# The agent uses InferenceClientModel with Qwen 2.5
# as its brain. It reasons about which tool to call next based on
# the task description and the result of each tool call.
# ============================================================

from smolagents import CodeAgent, InferenceClientModel

# Agent "brain" — a large free model on Hugging Face Inference API
agent_model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-72B-Instruct",
    token=HF_TOKEN
)

# Create the agent and give it all 4 tools
agent = CodeAgent(
    tools=[
        resolve_and_save_channel,
        fetch_and_save_videos,
        fetch_analyze_save_comments,
        generate_sentiment_report
    ],
    model=agent_model,
    max_steps=10,           # max tool calls before agent stops
    verbosity_level=1       # show each step the agent takes
)

print("Agent is ready!")
print("Model  : Qwen/Qwen2.5-72B-Instruct (via HF Inference API)")
print("Tools  : 4 tools loaded")


Agent is ready!
Model  : Qwen/Qwen2.5-72B-Instruct (via HF Inference API)
Tools  : 4 tools loaded


##  — Run the Agent

In [ ]:
# ============================================================
# CELL 7: Run the Agent
#
# Enter a YouTube channel below.
# The agent will automatically call the 4 tools in order.
# ============================================================


channel_input = input("Enter YouTube channel (handle / URL / ID): ").strip()


# The agent reads this and decides how to call the tools.
task = f"""
You are a YouTube sentiment analysis agent.
Analyze the YouTube channel: '{channel_input}'

Follow these steps in exact order:

Step 1 — Call resolve_and_save_channel(channel_input='{channel_input}')
         The result is a pipe-separated string: SUCCESS|channel_id|title|uploads_playlist_id
         Parse it to get channel_id and uploads_playlist_id.

Step 2 — Call fetch_and_save_videos(channel_id=<from step 1>, uploads_playlist_id=<from step 1>)
         The result is: SUCCESS|video_id1,video_id2,...|N videos saved
         Parse it to get the comma-separated video_ids string.

Step 3 — Call fetch_analyze_save_comments(video_ids_str=<video_ids from step 2>)
         This fetches comments, runs sentiment analysis, and saves everything.

Step 4 — Call generate_sentiment_report(channel_id=<from step 1>)
         This generates CSV reports and returns the final summary.

Return the final summary text from Step 4.
"""

print("Starting agent analysis...")
print("=" * 60)
final_result = agent.run(task)
print("=" * 60)
print("\nAgent finished!")


Enter YouTube channel (handle / URL / ID): https://www.youtube.com/@khanacademy
Starting agent analysis...


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are a YouTube sentiment analysis agent.                                                                     │
│ Analyze the YouTube channel: 'https://www.youtube.com/@khanacademy'                                             │
│                                                                                                                 │
│ Follow these steps in exact order:                                                                              │
│                                                                                                                 │
│ Step 1 — Call resolve_and_save_channel(channel_input='https://www.youtube.com/@khanacademy')                    │
│          The result is a pipe-separated string: SUCCESS|channel_id|title|uploads_playlist_id                    │
│          Parse it to get channel_id and uploads_playlist_id.                                                    │
│                                                                                                                 │
│ Step 2 — Call fetch_and_save_videos(channel_id=<from step 1>, uploads_playlist_id=<from step 1>)                │
│          The result is: SUCCESS|video_id1,video_id2,...|N videos saved                                          │
│          Parse it to get the comma-separated video_ids string.                                                  │
│                                                                                                                 │
│ Step 3 — Call fetch_analyze_save_comments(video_ids_str=<video_ids from step 2>)                                │
│          This fetches comments, runs sentiment analysis, and saves everything.                                  │
│                                                                                                                 │
│ Step 4 — Call generate_sentiment_report(channel_id=<from step 1>)                                               │
│          This generates CSV reports and returns the final summary.                                              │
│                                                                                                                 │
│ Return the final summary text from Step 4.                                                                      │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = resolve_and_save_channel(channel_input='https://www.youtube.com/@khanacademy')                          
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Channel saved to database!
  Name        : Khan Academy
  Subscribers : 9,350,000


...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(report_result)                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 
============================================================
         YOUTUBE CHANNEL SENTIMENT REPORT
============================================================

  Channel Name    : Khan Academy
  Subscribers     : 9,350,000
  Videos Checked  : 10
  Total Comments  : 32

  ---- Sentiment Breakdown ----
  Positive  :    16 comments  (50.0%)
  Negative  :    16 comments  (50.0%)
  Neutral   :     0 comments  (0.0%)

  Avg Confidence  : 0.9242
  Verdict         : MOSTLY POSITIVE  - Viewers generally enjoy this channel.

  ---- Files Saved ----
  video_sentiment_report.csv
  channel_summary_report.csv
  all_comments_with_sentiment.csv

============================================================

[Step 5: Duration 2.01 seconds| Input tokens: 15,874 | Output tokens: 880]


Agent finished!


## — View Reports in Notebook

In [ ]:

# : View Video-wise Report in the Notebook


import pandas as pd

print("\n--- VIDEO-WISE SENTIMENT REPORT ---")
df_v = pd.read_csv("video_sentiment_report.csv")
# will Show only the most useful columns
cols_to_show = ["title", "total_comments", "positive_count",
                "negative_count", "neutral_count", "positive_pct", "avg_sentiment_score"]
display(df_v[cols_to_show])

print("\n--- CHANNEL SUMMARY ---")
df_c = pd.read_csv("channel_summary_report.csv")
display(df_c)



--- VIDEO-WISE SENTIMENT REPORT ---


,title,total_comments,positive_count,negative_count,neutral_count,positive_pct,avg_sentiment_score
0,"Coming this summer! Khan Academy—still free, n...",3,2.0,1.0,0.0,66.67,0.8952
1,The intersection of creativity and biology | K...,2,1.0,1.0,0.0,50.00,0.9958
2,Real tools for real classrooms: A direct look ...,8,3.0,5.0,0.0,37.50,0.9104
3,What happens to the workforce?,1,0.0,1.0,0.0,0.00,0.9994
4,The world ocean | Middle school Earth and spac...,0,0.0,0.0,0.0,NaN,NaN
5,See what happens when we build tools for how t...,2,2.0,0.0,0.0,100.00,0.9973
6,Too many students start behind for reasons tha...,3,2.0,1.0,0.0,66.67,0.8245
7,Freshwater use and conservation | Middle schoo...,0,0.0,0.0,0.0,NaN,NaN
8,Will AI replace teachers?,9,4.0,5.0,0.0,44.44,0.9060
9,Alive for one minute and already asking how to...,4,2.0,2.0,0.0,50.00,0.9981



--- CHANNEL SUMMARY ---


,channel_title,subscriber_count,videos_analyzed,total_comments,positive_comments,negative_comments,neutral_comments,avg_sentiment_score,positive_pct
0,Khan Academy,9350000,10,32,16.0,16.0,0.0,0.9242,50.0


##  Export SQL Dump

In [ ]:

#  Export SQL Dump


import mysql.connector
from datetime import date, datetime


def sql_value(value):
    """Convert Python value into SQL-safe text."""
    if value is None:
        return "NULL"

    if isinstance(value, (datetime, date)):
        return f"'{value}'"

    if isinstance(value, str):
        value = value.replace("\\", "\\\\").replace("'", "''")
        return f"'{value}'"

    return str(value)


def export_sql_dump(file_name="sentiment_analysis_dump.sql"):
    conn = mysql.connector.connect(**DB_CONFIG)
    cur = conn.cursor()

    tables = ["channels", "videos", "comments"]
    lines = [
        "-- YouTube Sentiment Analysis SQL Dump",
        ""
    ]

    for table in tables:
        lines.append(f"-- TABLE: {table}")
        lines.append(f"DROP TABLE IF EXISTS {table};")

        cur.execute(f"SHOW CREATE TABLE {table}")
        create_sql = cur.fetchone()[1]
        lines.append(create_sql + ";")

        cur.execute(f"SELECT * FROM {table}")
        columns = [col[0] for col in cur.description]

        for row in cur.fetchall():
            cols = ", ".join(columns)
            vals = ", ".join(sql_value(v) for v in row)
            lines.append(f"INSERT INTO {table} ({cols}) VALUES ({vals});")

        lines.append("")

    cur.close()
    conn.close()

    with open(file_name, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"SQL dump saved: {file_name}")


export_sql_dump()


SQL dump saved: sentiment_analysis_dump.sql


##  Download All Output Files

In [ ]:

# CELL 10: Download All Output Files (Colab only)


from google.colab import files

output_files = [
    "video_sentiment_report.csv",
    "channel_summary_report.csv",
    "all_comments_with_sentiment.csv",
    "sentiment_analysis_dump.sql"
]

for fname in output_files:
    try:
        files.download(fname)
        print(f"Downloaded: {fname}")
    except Exception as e:
        print(f"Could not download {fname}: {e}")

print("\nDone! All project deliverables are ready.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: video_sentiment_report.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloaded: all_comments_with_sentiment.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: sentiment_analysis_dump.sql

Done! All project deliverables are ready.
